# Filtered, Not Faked (fast run)
Do beauty filters make real faces get flagged as deepfakes, and does it hit darker skin harder?

Runtime: **GPU (T4)**. Runtime > Change runtime type > T4 GPU.

Fast design: 75 images per race group x 7 groups = 525 faces; control + strengths 0.5 and 1.0 for 3 filters = 7 versions per face = 3,675 detector passes. Expect under 10 minutes on a T4.

## 1. Get the code and images
Upload `filtered-not-faked.zip` (it already contains `data/fairface/`, so nothing is downloaded from FairFace).

In [ ]:
from google.colab import files
up = files.upload()
!unzip -q -o filtered-not-faked.zip
%cd filtered-not-faked
import torch; print('GPU:', torch.cuda.is_available())

In [ ]:
!pip install -q transformers statsmodels opencv-python-headless

## 2. Timing test (20 images)
Prints seconds per image. Multiply by 525 for the full run. If the estimate is over 45 minutes, set `N_PER_GROUP = 50` below.

In [ ]:
DET = 'prithivMLmods/Deep-Fake-Detector-v2-Model'
!python run_experiment.py --source folder --path data/fairface --n-per-group 75 --max-images 20 --detectors {DET} --out results/timing

## 3. Real run

In [ ]:
N_PER_GROUP = 75
!python run_experiment.py --source folder --path data/fairface --n-per-group {N_PER_GROUP} --detectors {DET} --out results/fairface

## 4. Analyse

In [ ]:
!python analyze.py results/fairface
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/fairface/fig_*.png')):
    display(Image(p))

In [ ]:
import pandas as pd
pd.read_csv('results/fairface/regression.csv').query("term == 'strength:ita_z'")

In [ ]:
for p in sorted(glob.glob('results/fairface/examples/*.jpg'))[:4]:
    display(Image(p, width=600))

## 5. Download results

In [ ]:
!zip -qr results_fairface.zip results/fairface
files.download('results_fairface.zip')